# 00 Data Processing


In [3]:
%load_ext autoreload
%autoreload 2

import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from numpy import linalg as la

from config import (
    DATA_DIR,
    MOLECULE_DIR,
    PARAMETERS_DIR,
    RAW_MATRICES_DIR,
    PROCESSED_DATAFRAMES_DIR
)
from notebook_utils.general import complex_matrix, read_jsonl

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Statevector Dataset

In [4]:
SV_HAMILTONIAN_DIR = RAW_MATRICES_DIR / "SV_hamiltonian"
SV_SPIN_DIR = RAW_MATRICES_DIR / "SV_spin"
PYSCF_CASCI_ENERGIES_PATH = RAW_MATRICES_DIR / "pyscf_casci_energies.jsonl"
def load_sv_hamiltonian_records(path):
    records = []

    for file in sorted(path.glob("*.jsonl")):
        for row in read_jsonl(file):
            H = complex_matrix(row["h_matrix_real"], row["h_matrix_imag"])
            S = complex_matrix(row["s_matrix_real"], row["s_matrix_imag"])

            records.append({
                "molecule": row["molecule"],
                "active_space": row["active_space"],
                "ansatz": row["ansatz"],
                "expansion": row["expansion"],
                "H": H,
                "S": S,
                "qse_dim": H.shape[0],
            })

    return pd.DataFrame(records)


def load_spin_records(path):
    records = []

    for file in sorted(path.glob("*.jsonl")):
        for row in read_jsonl(file):
            S2 = complex_matrix(row["z_matrix_real"], row["z_matrix_imag"])

            records.append({
                "molecule": row["molecule"],
                "active_space": row["active_space"],
                "ansatz": row["ansatz"],
                "expansion": row["expansion"],
                "S2": S2
            })

    return pd.DataFrame(records)

def load_pyscf_casci_energy_records(path):
    records = []

    for row in read_jsonl(path):
        records.append({
            "molecule": row["molecule"],
            "active_space": row["active_space"],
            "spin_type": row["spin_type"],
            "pyscf_casci_energies": row["casci_energies"],
            "pyscf_casci_pvec": row["casci_pvec"],
        })

    return pd.DataFrame(records)


df_sv_hamiltonian = load_sv_hamiltonian_records(SV_HAMILTONIAN_DIR)
df_sv_spin = load_spin_records(SV_SPIN_DIR)
df_pyscf_casci = load_pyscf_casci_energy_records(PYSCF_CASCI_ENERGIES_PATH)


In [5]:
# Compress the PYSCF Triplet energies
def average_triplet_sector_energies(casci_energies):
    average_energies = []
    sector_diffs = []

    for nth_energies in casci_energies:
        energies = list(nth_energies.values())
        average_energies.append(float(np.mean(energies)))
        sector_diffs.append(float(np.ptp(energies)))

    return average_energies, sector_diffs


new_casci_energies = []
triplet_sector_diffs = []

for _, row in df_pyscf_casci.iterrows():
    if row["spin_type"] == "triplet_all":
        energies, sector_diffs = average_triplet_sector_energies(
            row["pyscf_casci_energies"]
        )
        triplet_sector_diffs.extend(sector_diffs)
    else:
        energies = row["pyscf_casci_energies"]

    new_casci_energies.append(energies)


df_pyscf_casci["pyscf_casci_energies"] = new_casci_energies

max_triplet_sector_diff = max(triplet_sector_diffs, default=0.0)
print(f"Maximum PYSCF triplet sector energy difference: {max_triplet_sector_diff:.12e} Ha")


Maximum PYSCF triplet sector energy difference: 1.003218130791e-08 Ha


In [6]:
df_sv = df_sv_hamiltonian.merge(
    df_sv_spin,
    on=["molecule", "active_space", "ansatz", "expansion"],
    how="left",
)

df_sv = df_sv.merge(
    df_pyscf_casci,
    left_on=["molecule", "active_space", "expansion"],
    right_on=["molecule", "active_space", "spin_type"],
    how="left",
).drop(columns="spin_type")

df_sv = df_sv.sort_values(
    ["molecule", "active_space", "ansatz", "expansion"]
).reset_index(drop=True)

In [7]:
required_columns = ["H", "S", "S2", "pyscf_casci_energies", "pyscf_casci_pvec"]

df_sv_complete = df_sv.dropna(subset=required_columns).reset_index(drop=True)

PROCESSED_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
SV_DATAFRAME_PATH = PROCESSED_DATAFRAMES_DIR / "sv_qse_data.pkl"

df_sv_complete.to_pickle(SV_DATAFRAME_PATH)

df_sv_complete


,molecule,active_space,ansatz,expansion,H,S,qse_dim,S2,pyscf_casci_energies,pyscf_casci_pvec
0,Acetamide,2e2o,1UpCCGSDSinglet,singlet,"[[(-802.5663414572622+0j), (-4.706692634862638...","[[(3.9097041888243123+0j), (0.0229245233142809...",4,"[[(1.27675647831893e-14+0j), (-4.3368086899420...","[-205.29449911826808, -204.8258180441668, -204...","{'0': [-0.15005494805049122, 0, -0.00628487098..."
1,Acetamide,2e2o,1UpCCGSDSinglet,triplet_all,"[[(-0.008069881124952466+0j), 0j, 0j, (0.19302...","[[(3.9350963402490224e-05+0j), 0j, 0j, (-0.000...",12,"[[(7.870192680212162e-05+0j), 0j, 0j, (-0.0018...",[-205.07455054622508],"{'0_0': [-4.001328838134542e-16, 0, -0.7071067..."
2,Acetamide,2e2o,UCCGSD,singlet,"[[(-802.5658955566977+0j), (-4.714686911530125...","[[(3.909702017207626+0j), (0.02296346038353209...",4,"[[(1.2961853812498703e-14+0j), (-4.33680868994...","[-205.29449911826808, -204.8258180441668, -204...","{'0': [-0.15005494805049122, 0, -0.00628487098..."
3,Acetamide,2e2o,UCCGSD,triplet_all,"[[(-0.008097333197893943+0j), 0j, 0j, (0.19335...","[[(3.9484827272451284e-05+0j), 0j, 0j, (-0.000...",12,"[[(7.896965454190497e-05+0j), 0j, 0j, (-0.0018...",[-205.07455054622508],"{'0_0': [-4.001328838134542e-16, 0, -0.7071067..."
4,Acetamide,2e2o,UCCSD,singlet,"[[(-802.5658955566977+0j), (-4.714686911530125...","[[(3.909702017207626+0j), (0.02296346038353209...",4,"[[(1.2961853812498703e-14+0j), (-4.33680868994...","[-205.29449911826808, -204.8258180441668, -204...","{'0': [-0.15005494805049122, 0, -0.00628487098..."
...,...,...,...,...,...,...,...,...,...,...
1563,Uracil,6e6o,UCCGSD,triplet_all,"[[(-3.056612740692799e-10+0j), 0j, 0j, 0j, 0j,...","[[(7.535361223887094e-13+0j), 0j, 0j, 0j, 0j, ...",108,"[[(1.9750867608081535e-12+0j), 0j, 0j, 0j, 0j,...","[-406.9563843369734, -406.90378655913327, -406...","{'0_0': [-5.551115123125783e-17, 0, -0.0006048..."
1564,Uracil,6e6o,UCCSD,singlet,"[[(-1628.146797538434+0j), 0j, 0j, 0j, 0j, 0j,...","[[(3.9991745183913965+0j), 0j, 0j, 0j, 0j, 0j,...",36,"[[(2.8960363056024663e-07+0j), 0j, 0j, 0j, 0j,...","[-407.12116588854, -406.889567697793, -406.821...","{'0': [0.00023156095438382814, 0, 4.8560227557..."
1565,Uracil,6e6o,UCCSD,triplet_all,"[[(-1.0753353762993356e-10+0j), 0j, 0j, 0j, 0j...","[[(2.6509350270487175e-13+0j), 0j, 0j, 0j, 0j,...",108,"[[(6.958877918350481e-13+0j), 0j, 0j, 0j, 0j, ...","[-406.9563843369734, -406.90378655913327, -406...","{'0_0': [-5.551115123125783e-17, 0, -0.0006048..."
1566,Uracil,6e6o,UCCSDSinglet,singlet,"[[(-1628.1503087714625+0j), 0j, 0j, 0j, 0j, 0j...","[[(3.9991831439323717+0j), 0j, 0j, 0j, 0j, 0j,...",36,"[[(8.385509530479084e-08+0j), 0j, 0j, 0j, 0j, ...","[-407.12116588854, -406.889567697793, -406.821...","{'0': [0.00023156095438382814, 0, 4.8560227557..."


### Shots dataset

In [8]:
SHOTS_HAMILTONIAN_DIR = RAW_MATRICES_DIR / "shots_hamiltonian"


def load_shots_hamiltonian_records(path):
    records = []

    for file in sorted(path.glob("*.jsonl")):
        for row in read_jsonl(file):
            H = complex_matrix(row["h_matrix_real"], row["h_matrix_imag"])
            S = complex_matrix(row["s_matrix_real"], row["s_matrix_imag"])

            records.append({
                "molecule": row["molecule"],
                "active_space": row["active_space"],
                "ansatz": row["ansatz"],
                "expansion": row["expansion"],
                "sample_key": row["sample_key"],
                "n_shots": row["n_shots"],
                "repeat": row["repeat"],
                "H_shots": H,
                "S_shots": S,
                "qse_dim": H.shape[0],
            })

    return pd.DataFrame(records)


df_shots_hamiltonian = load_shots_hamiltonian_records(SHOTS_HAMILTONIAN_DIR)

df_shots_sv = df_sv_hamiltonian.rename(columns={
    "H": "H_sv",
    "S": "S_sv",
    "qse_dim": "qse_dim_sv",
})

df_shots = df_shots_hamiltonian.merge(
    df_shots_sv,
    on=["molecule", "active_space", "ansatz", "expansion"],
    how="left",
)

df_shots = df_shots.sort_values(
    ["molecule", "active_space", "ansatz", "expansion", "n_shots", "repeat"]
).reset_index(drop=True)


In [9]:
required_columns = ["H_shots", "S_shots", "H_sv", "S_sv"]

df_shots_complete = df_shots.dropna(subset=required_columns).reset_index(drop=True)

PROCESSED_DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
SHOTS_DATAFRAME_PATH = PROCESSED_DATAFRAMES_DIR / "shots_qse_data.pkl"

df_shots_complete.to_pickle(SHOTS_DATAFRAME_PATH)

df_shots_complete


,molecule,active_space,ansatz,expansion,sample_key,n_shots,repeat,H_shots,S_shots,qse_dim,H_sv,S_sv,qse_dim_sv
0,Acetamide,2e2o,UCCSD,singlet,shots_1000_0,1000,0,"[[(-799.3445190906648+0j), (7.286636362415686+...","[[(3.894+0j), (-0.035500000000000004-0.0315j),...",4,"[[(-802.5658955566977+0j), (-4.714686911530125...","[[(3.909702017207626+0j), (0.02296346038353209...",4
1,Acetamide,2e2o,UCCSD,singlet,shots_1000_1,1000,1,"[[(-799.3402010752859+0j), (-2.156579216824391...","[[(3.894+0j), (0.010499999999999995+0.00774999...",4,"[[(-802.5658955566977+0j), (-4.714686911530125...","[[(3.909702017207626+0j), (0.02296346038353209...",4
2,Acetamide,2e2o,UCCSD,singlet,shots_1000_2,1000,2,"[[(-806.3234685164332+0j), (-4.82329623630002-...","[[(3.928+0j), (0.02350000000000002+0.011000000...",4,"[[(-802.5658955566977+0j), (-4.714686911530125...","[[(3.909702017207626+0j), (0.02296346038353209...",4
3,Acetamide,2e2o,UCCSD,singlet,shots_1000_3,1000,3,"[[(-805.091173493782+0j), (-4.774631858024356-...","[[(3.9219999999999997+0j), (0.0232500000000000...",4,"[[(-802.5658955566977+0j), (-4.714686911530125...","[[(3.909702017207626+0j), (0.02296346038353209...",4
4,Acetamide,2e2o,UCCSD,singlet,shots_1000_4,1000,4,"[[(-799.3455465676125+0j), (-8.981013971140694...","[[(3.894+0j), (0.04375000000000001-0.044500000...",4,"[[(-802.5658955566977+0j), (-4.714686911530125...","[[(3.909702017207626+0j), (0.02296346038353209...",4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
78395,Uracil,6e6o,UCCSDSinglet,triplet_all,shots_1000000_5,1000000,5,"[[(-6.964897541195114e-05+0j), (0.180803532647...","[[(2.7755575615628914e-17+0j), (-0.00044512371...",108,"[[(-1.2670398064074107e-10+0j), 0j, 0j, 0j, 0j...","[[(3.123612479782878e-13+0j), 0j, 0j, 0j, 0j, ...",108
78396,Uracil,6e6o,UCCSDSinglet,triplet_all,shots_1000000_6,1000000,6,"[[(-1.948318953282069e-05+0j), (0.010756626273...","[[(2.7755575615628914e-17+0j), (-2.65165042944...",108,"[[(-1.2670398064074107e-10+0j), 0j, 0j, 0j, 0j...","[[(3.123612479782878e-13+0j), 0j, 0j, 0j, 0j, ...",108
78397,Uracil,6e6o,UCCSDSinglet,triplet_all,shots_1000000_7,1000000,7,"[[(-2.8408108221356088e-06+0j), (-0.0099200750...","[[(2.7755575615628914e-17+0j), (2.474873734154...",108,"[[(-1.2670398064074107e-10+0j), 0j, 0j, 0j, 0j...","[[(3.123612479782878e-13+0j), 0j, 0j, 0j, 0j, ...",108
78398,Uracil,6e6o,UCCSDSinglet,triplet_all,shots_1000000_8,1000000,8,"[[(1.7512099191208108e-05+0j), (-0.00899883163...","[[(8.326672684688674e-17+0j), (2.1920310216795...",108,"[[(-1.2670398064074107e-10+0j), 0j, 0j, 0j, 0j...","[[(3.123612479782878e-13+0j), 0j, 0j, 0j, 0j, ...",108
